In [ ]:
# 독립변수에 범주형 자료가 있을 경우, onehotencoder를 이용해서 예측
# MultipleLinearRegressionData.csv 파일을 이용해서 공부시간,결석,장소를 입력해서 성적을 예측

In [39]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression 
from sklearn.preprocessing import PolynomialFeatures  # 다항특성 자동생성
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.preprocessing import StandardScaler  # 정규화클래스
from sklearn.model_selection import train_test_split

In [59]:
df = pd.read_csv('../data/MultipleLinearRegressionData.csv') 
 # CSV 파일을 읽어 데이터프레임 df를 만듭니다.
data = df[df.columns[:-2]].values  # 뒤에서 두 번째 열 앞까지(숫자형 독립변수)만 넘파이 배열로 꺼냅니다.
label = df['score'].values  # 맨 마지막 열(정답/타겟 점수)을 넘파이 배열로 꺼냅니다.
label

array([ 10,   8,  14,  26,  22,  30,  42,  48,  38,  58,  60,  72,  62,
        68,  72,  58,  76,  86,  90, 100])

In [ ]:
from sklearn.preprocessing import OneHotEncoder  
# 범주형 문자를 0과 1의 더미 열로 바꾸는 클래스를 가져옵니다.
place = df['place'].to_numpy().reshape(-1, 1)  
# 'place' 열을 넘파이 배열로 바꾼 뒤 2차원 (행 개수, 1) 형태로 만듭니다.
encoder = OneHotEncoder(sparse_output=False, drop='first')  
# 일반 넘파이 배열로 반환하고 첫 범주를 제외해 다중공선성을 줄이는 인코더를 만듭니다[cite: 1].
# 첫번째 범주는 알파벳 순서로 맨앞의 범주를 말함(home, library, cafe이면, cafe가 첫 범주임)
result = encoder.fit_transform(place) 
# 장소 데이터를 학습하고 원핫인코딩된 배열로 변환합니다[cite: 1].
total_data = np.concatenate((data, result), axis=1)  
# 숫자형 특성(data)과 인코딩된 장소 열(result)을 좌우(열 방향)로 합칩니다[cite: 1].

In [ ]:
# 3개 범주 -> 2개의 더미 : home, library, cafe는 순서대로 1,0 0,1 0,0
# 5개 범주 -> 4개 더미 : 알파베 순서대로 (0,0,0,0) (1,0,0,0) (0,1,0,0) (0,0,1,0) (0,0,0,1)가 됨

In [48]:
train_data, test_data, train_label, test_label = train_test_split(
    total_data, label, random_state=42
)  # 전체 특성과 정답을 재현 가능하게(random_state=42) 훈련 세트와 테스트 세트로 나눕니다[cite: 1].

In [63]:
lr = LinearRegression()  # 규제가 없는 기본 선형회귀 모델 객체를 만듭니다[cite: 1].
lr.fit(train_data, train_label)  # 훈련 데이터와 훈련 정답으로 선형회귀 모델을 학습시킵니다[cite: 1].

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](4,)","[10.67,-1.18,-7.33,-3.1 ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,3.587
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(4)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](4,)","[8.67,3.75,1.66,1.17]"


In [62]:
print(lr.score(train_data, train_label))  # 훈련 데이터에 대한 결정계수(R^2) 점수를 출력합니다[cite: 1].
print(lr.score(test_data, test_label))  # 테스트 데이터에 대한 결정계수(R^2) 점수를 출력합니다[cite: 1].

0.9770255436151457
0.9551480880200326


In [61]:
predict = lr.predict(test_data)  # 테스트 데이터 전체에 대한 예측 점수를 계산합니다[cite: 1].
print('예측점수 : ', predict)  # 테스트 데이터 예측 점수 배열을 출력합니다[cite: 1].

예측점수 :  [-1.96441609 84.65989422 65.11068424  8.54709764 38.32995382]


In [ ]:
# 모형을 이용한 실제 예측(공부시간, 결석횟수, 장소 : 7,1,Cafe)
# 자료입력 및 (파일에서) 적용 -> 예측에 맞는 형태로 변경 -> 예측

In [ ]:
# 자료 입력 및 자료형태 조정

in_data = input('공부시간, 결석횟수, 장소를 입력하세요. (예: 7,1,cafe)')  
# 한 줄 문자열로 공부시간, 결석횟수, 장소를 입력받습니다[cite: 1].
in_data = in_data.split(',')  
# 쉼표(,) 기준으로 문자열을 잘라 리스트로 만듭니다[cite: 1].
out_data = []  # 모델 입력용 특성 4개를 순서대로 담을 빈 리스트를 만듭니다[cite: 1].
out_data.append(float(in_data[0]))  
# 첫 번째 값(공부시간)을 실수(float)로 바꿔 추가합니다[cite: 1].
out_data.append(float(in_data[1])) 
# 두 번째 값(결석횟수)을 실수(float)로 바꿔 추가합니다[cite: 1].
if in_data[2] == 'Home':  # 장소가 'Home'이면[cite: 1]
    out_data.append(1.0)  # 첫 번째 장소 더미 변수에 1.0을 넣고[cite: 1]
    out_data.append(0.0)  # 두 번째 장소 더미 변수에 0.0을 넣습니다[cite: 1].
elif in_data[2] == 'Library':  # 장소가 'Library'이면[cite: 1]
    out_data.append(0.0)  # 첫 번째 장소 더미 변수에 0.0을 넣고[cite: 1]
    out_data.append(1.0)  # 두 번째 장소 더미 변수에 1.0을 넣습니다[cite: 1].
elif in_data[2] == 'cafe':  # 장소가 'Cafe'이면 (drop='first' 기준선)[cite: 1]
    out_data.append(0.0)  # 첫 번째 장소 더미 변수에 0.0을 넣고[cite: 1]
    out_data.append(0.0)  # 두 번째 장소 더미 변수에 0.0을 넣습니다[cite: 1].

In [ ]:
# 모형 예측
result = lr.predict([out_data])  
# 1건 데이터를 2차원 리스트([out_data])로 감싸 선형회귀 모델로 예측합니다[cite: 1].
print(f"1개 예측점수 : {result[0]:.2f}점")  
# 예측된 첫 번째 값을 소수점 둘째 자리까지 점수로 출력합니다[cite: 1].

1개 예측점수 : 77.06점


In [ ]:
# 다항식과 표준화를 반영한 릿지모형으로 추정

In [ ]:
# 다항식 반영

poly = PolynomialFeatures(degree=5, include_bias=False)  
# 5차 다항식 조합을 만들되 절편용 1열(bias)은 제외하는 변환기를 만듭니다[cite: 1].
poly.fit(train_data)  # 훈련 데이터의 특성 구조를 기준으로 다항 변환 규칙을 맞춥니다[cite: 1].
train_poly = poly.transform(train_data)  # 훈련 데이터를 다항 특성 행렬로 변환합니다[cite: 1].
test_poly = poly.transform(test_data)  # 테스트 데이터를 동일한 규칙으로 다항 특성 행렬로 변환합니다[cite: 1].

In [ ]:
# 표준화 반영

ss = StandardScaler()  # 평균 0, 표준편차 1로 표준화하는 스케일러 객체를 만듭니다[cite: 1].
ss.fit(train_poly)  # 훈련 다항 특성의 평균과 표준편차를 계산해 맞춥니다[cite: 1].
train_scaled = ss.transform(train_poly)  # 훈련 다항 특성을 표준화합니다[cite: 1].
test_scaled = ss.transform(test_poly)  # 테스트 다항 특성을 훈련 기준 평균·표준편차로 표준화합니다[cite: 1].

In [ ]:
# 릿지모형

alpha_list = [0.001, 0.01, 0.1, 1, 10, 100]  
# 시험해 볼 규제 강도 alpha 후보 값들을 리스트로 둡니다[cite: 1].
train_score = []  # 각 alpha일 때의 훈련 점수를 모을 리스트입니다[cite: 1].
test_score = []  # 각 alpha일 때의 테스트 점수를 모을 리스트입니다[cite: 1].
for alpha in alpha_list:  # alpha 후보 값을 하나씩 꺼내 반복합니다[cite: 1].
    ridge = Ridge(alpha=alpha)  # 현재 alpha 값이 적용된 릿지 회귀 모델을 생성합니다[cite: 1].
    ridge.fit(train_scaled, train_label)  
    # 표준화된 훈련 데이터와 정답으로 릿지 모델을 학습시킵니다[cite: 1].
    train_score.append(ridge.score(train_scaled, train_label))  
    # 표준화된 훈련 세트의 R^2 점수를 기록합니다[cite: 1].
    test_score.append(ridge.score(test_scaled, test_label))  
    # 표준화된 테스트 세트의 R^2 점수를 기록합니다[cite: 1].
    print('alpha:',alpha)
    print('train:',train_score[-1])
    print('test:',test_score[-1])

alpha: 0.001
train: 0.9999969055028449
test: -19.841434685303422
alpha: 0.01
train: 0.9998812237349386
test: -8.290516058463199
alpha: 0.1
train: 0.9989159986007818
test: -1.8102168848321725
alpha: 1
train: 0.9848303271412874
test: -5.615895485914004
alpha: 10
train: 0.8922999589881696
test: -11.16168642137038
alpha: 100
train: 0.6965671459177547
test: -17.00372787951453


In [ ]:
plt.plot(np.log10(alpha_list), train_score, marker='o')  
# x축을 log10(alpha)로 두고 훈련 점수 변화를 점 마커 선으로 그립니다[cite: 1].
plt.plot(np.log10(alpha_list), test_score, marker='o')  
# 같은 x축 기준으로 테스트 점수 변화를 점 마커 선으로 그립니다[cite: 1].
plt.xlabel('log10(alpha)')  # x축 이름을 'log10(alpha)'로 표시합니다[cite: 1].
plt.ylabel('R2 Score')  # y축 이름을 'R2 Score'로 표시합니다[cite: 1].
plt.legend(['Train', 'Test'])  # 두 선이 각각 훈련(Train)과 테스트(Test)임을 범례로 표시합니다[cite: 1].
plt.grid()  # 그래프에 격자선을 추가해 값을 읽기 쉽게 합니다[cite: 1].
plt.show()  # 완성된 그래프를 화면에 출력합니다[cite: 1].

In [56]:
ridge = Ridge(alpha=1)  # 규제 강도 alpha를 1로 지정한 릿지 모델을 만듭니다[cite: 1].
ridge.fit(train_scaled, train_label)  # 표준화된 훈련 데이터로 최종 릿지 모델을 학습시킵니다[cite: 1].
print('[최종 모델 성능]')  # 최종 모델 평가 결과 제목을 출력합니다[cite: 1].
print('훈련 정확도:', ridge.score(train_scaled, train_label))  # 최종 릿지 모델의 훈련 세트 R^2 점수를 출력합니다[cite: 1].
print('테스트 정확도:', ridge.score(test_scaled, test_label))  # 최종 릿지 모델의 테스트 세트 R^2 점수를 출력합니다[cite: 1].

[최종 모델 성능]
훈련 정확도: 0.9848303271412874
테스트 정확도: -5.615895485914004


In [57]:
result = ridge.predict(test_scaled)  # 최종 릿지 모델로 테스트 세트의 점수를 예측합니다[cite: 1].
mae = mean_absolute_error(test_label, result)  # 실제 테스트 정답과 예측값 차이의 절댓값 평균(MAE)을 계산합니다[cite: 1].
print('평균 절대 오차(MAE):', mae)  # 평균적으로 몇 점 정도 오차가 나는지 MAE 값을 출력합니다[cite: 1].

평균 절대 오차(MAE): 66.71842905136684
1.0
-101.06534049933485


In [ ]:
# 규제를 제거한 다항선형회귀
lr = LinearRegression()  # 규제가 없는 선형회귀 모델을 새로 만듭니다[cite: 1].
lr.fit(train_poly, train_label)  # 다항 특성(train_poly)으로 규제 없이 학습시킵니다[cite: 1].
print(lr.score(train_poly, train_label))  # 규제 없는 다항 회귀의 훈련 점수를 출력합니다[cite: 1].
print(lr.score(test_poly, test_label))  # 규제 없는 다항 회귀의 테스트 점수를 출력합니다[cite: 1].